In [1]:
import torch
print("MPS disponível (aceleração Apple Silicon):", torch.backends.mps.is_available())

from transformers import pipeline

gerador = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
    device="mps" if torch.backends.mps.is_available() else "cpu",
)

resposta = gerador(
    [{"role": "user", "content": "Explica em 2 frases o que é um keylogger."}],
    max_new_tokens=100,
)
print(resposta[0]["generated_text"][-1]["content"])

MPS disponível (aceleração Apple Silicon): True


/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use mps


Um keylogger é uma ferramenta maliciosa ou malware que captura e registra todas as teclas pressionadas por um usuário em um computador. Ele pode ser usado para rastrear atividades de escrita ou interação com sistemas digitais, como redes sociais ou programas de negócios.


In [2]:
import joblib
import json
import pandas as pd

rf = joblib.load("models/random_forest_keylogger_final.joblib")

with open("models/features_keylogger.json") as f:
    features = json.load(f)

print("Modelo carregado. Número de features esperadas:", len(features))
print(features[:5], "...")

Modelo carregado. Número de features esperadas: 43
['Source Port', 'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets'] ...


In [3]:
from src.data_prep import load_raw, clean_dataset
from sklearn.model_selection import train_test_split

df = clean_dataset(load_raw("data/Keylogger_Detection.csv"))

X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_test_fs = X_test[features]

print("Dados de teste prontos:", X_test_fs.shape)

Dados de teste prontos: (33176, 43)


In [4]:
def gerar_relatorio(indice, X_dados, y_real, modelo_rf, gerador_llm):
    linha = X_dados.iloc[[indice]]

    previsao = modelo_rf.predict(linha)[0]
    probas = modelo_rf.predict_proba(linha)[0]
    idx_classe = list(modelo_rf.classes_).index(previsao)
    confianca = probas[idx_classe]

    # As 5 features que mais pesaram na decisão deste modelo (globalmente)
    top_features = pd.Series(modelo_rf.feature_importances_, index=X_dados.columns)
    top_features = top_features.sort_values(ascending=False).head(5)
    valores_top = linha[top_features.index].iloc[0]

    dados_relevantes = "\n".join(
        f"- {feat}: {valores_top[feat]:.2f}" for feat in top_features.index
    )

    prompt = f"""Escreve um relatório curto (máximo 4 frases) em português, para uma equipa de segurança, sobre a análise de um fluxo de rede.

Classificação do modelo: {previsao}
Confiança: {confianca:.1%}

Os 5 valores mais relevantes que o modelo usou para decidir:
{dados_relevantes}

Explica em linguagem simples o que isto significa e, se for Keylogger, recomenda uma ação."""

    resposta = gerador_llm(
        [{"role": "user", "content": prompt}],
        max_new_tokens=200,
    )

    texto_relatorio = resposta[0]["generated_text"][-1]["content"]

    real = y_real.iloc[indice]
    print(f"--- Fluxo #{indice} ---")
    print(f"Classe real: {real} | Previsão: {previsao} | Confiança: {confianca:.1%}")
    print()
    print(texto_relatorio)

    return texto_relatorio

# Testar com um fluxo real de keylogger
indice_teste = y_test.reset_index(drop=True)[y_test.reset_index(drop=True) == "Keylogger"].index[0]
gerar_relatorio(indice_teste, X_test_fs.reset_index(drop=True), y_test.reset_index(drop=True), rf, gerador)

--- Fluxo #4 ---
Classe real: Keylogger | Previsão: Keylogger | Confiança: 87.0%

O modelo detectou um ataque Keylogger com base nos seguintes valores:

- O "Source Port" é um número único identificando o ponto inicial da conexão.
- A "Flow IAT Min", "Flow IAT Max" e "Flow Duration" são medidas de tempo que mostram como as mensagens estão sendo enviadas.
- "Init_Win_bytes_forward" indica o tamanho inicial dos pacotes de entrada.

Com base nessas informações, é provável que haja um malicioso software instalado na máquina que está interrompendo os pacotes normais para capturar teclas digitadas. Esta situação deve ser investigada rapidamente e medidas preventivas tomadas.


'O modelo detectou um ataque Keylogger com base nos seguintes valores:\n\n- O "Source Port" é um número único identificando o ponto inicial da conexão.\n- A "Flow IAT Min", "Flow IAT Max" e "Flow Duration" são medidas de tempo que mostram como as mensagens estão sendo enviadas.\n- "Init_Win_bytes_forward" indica o tamanho inicial dos pacotes de entrada.\n\nCom base nessas informações, é provável que haja um malicioso software instalado na máquina que está interrompendo os pacotes normais para capturar teclas digitadas. Esta situação deve ser investigada rapidamente e medidas preventivas tomadas.'

In [5]:
y_test_reset = y_test.reset_index(drop=True)
X_test_reset = X_test_fs.reset_index(drop=True)

indice_benign = y_test_reset[y_test_reset == "Benign"].index[0]
gerar_relatorio(indice_benign, X_test_reset, y_test_reset, rf, gerador)

--- Fluxo #0 ---
Classe real: Benign | Previsão: Benign | Confiança: 60.0%

O modelo detectou um fluxo de rede como benigno com confiança alta. Os cinco valores mais importantes foram:

1. **Source Port**: 46996.00 - Este é o número da porta origem do tráfego.
2. **Flow IAT Min**: 3024.00 - O tempo médio entre requisições de dados (IAT) no início do fluxo foi de 3024 milésimos.
3. **Flow IAT Max**: 3024.00 - O mesmo valor para o IAT máximo no início do fluxo.
4. **Flow Duration**: 3024.00 - O tempo total do fluxo de 3024 milésimos.
5. **Init_Win_bytes_forward**: 1390.00 - Quantidade inicial de bytes recebidos durante o início do fluxo.

Este tipo de análise


'O modelo detectou um fluxo de rede como benigno com confiança alta. Os cinco valores mais importantes foram:\n\n1. **Source Port**: 46996.00 - Este é o número da porta origem do tráfego.\n2. **Flow IAT Min**: 3024.00 - O tempo médio entre requisições de dados (IAT) no início do fluxo foi de 3024 milésimos.\n3. **Flow IAT Max**: 3024.00 - O mesmo valor para o IAT máximo no início do fluxo.\n4. **Flow Duration**: 3024.00 - O tempo total do fluxo de 3024 milésimos.\n5. **Init_Win_bytes_forward**: 1390.00 - Quantidade inicial de bytes recebidos durante o início do fluxo.\n\nEste tipo de análise'

In [6]:
y_test_reset = y_test.reset_index(drop=True)
X_test_reset = X_test_fs.reset_index(drop=True)

def gerar_relatorio_v2(indice, X_dados, y_real, modelo_rf, gerador_llm, max_tokens=300):
    return gerar_relatorio(indice, X_dados, y_real, modelo_rf, gerador_llm)